# Pandas Class Notes: Data Exploration, Cleaning, Key Validation & Sampling

This notebook covers:
1. **Environment Setup & Data Loading**: Importing packages, reading CSV files with datetime parsing, and extracting date features.
2. **Metadata & Exploration**: Examining table structure, calculating summary ratios, and validating `event_id` uniqueness.
3. **Duplicate Detection & Cleaning**: Identifying duplicate event records and removing affected members across all relational tables.
4. **Key Validation**: Creating a reusable function `check_key` to test primary and candidate keys.
5. **Timeline Inspection**: Sorting and viewing chronological event history for a sample member.
6. **Cohort Sampling**: Drawing a 10% representative member sample and preserving referential integrity across tables.

## 1. Environment Setup & Library Imports
Import core libraries required for file path handling, numerical operations, and data manipulation.

In [1]:
# Import standard and third-party libraries
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

## 2. Load Datasets & Feature Engineering
Load `members`, `renewals`, and `events` datasets from CSV files with appropriate datetime parsing, and extract date features (`join_year`, `expiry_year`, `expiry_month`).

In [2]:
# Define directory path where CSV files reside
DATA_DIR = Path("../notebooks")
print("Available files in directory:", os.listdir(DATA_DIR))

# 1. Load members dataset with datetime parsing for join_date
members = pd.read_csv(
    DATA_DIR / "members.csv",
    parse_dates=["join_date"]
)

# 2. Load renewals dataset with datetime parsing for expiry and deadline dates
renewals = pd.read_csv(
    DATA_DIR / "renewal_opportunities.csv",
    parse_dates=["expiry_date", "renewal_deadline"]
)

# 3. Load member events dataset with datetime parsing for event_time
events = pd.read_csv(
    DATA_DIR / "member_events.csv",
    parse_dates=["event_time"]
)

# Feature Engineering: Extract year and month components for downstream temporal analysis
members["join_year"] = members["join_date"].dt.year
renewals["expiry_year"] = renewals["expiry_date"].dt.year
renewals["expiry_month"] = renewals["expiry_date"].dt.month

print(f"Loaded members:  {members.shape}")
print(f"Loaded renewals: {renewals.shape}")
print(f"Loaded events:   {events.shape}")

Available files in directory: ['DATA_DICTIONARY.md', 'Kalpa_Plus_Student.ipynb', 'members.csv', 'renewal_opportunities.csv', 'member_events.csv']


Loaded members:  (8580, 9)
Loaded renewals: (11580, 9)
Loaded events:   (152057, 10)


## 3. Initial Data Inspection
Preview the first few records of each loaded DataFrame to inspect column names and values.

In [3]:
# Preview the first 5 records of each DataFrame
print("Members Preview:")
display(members.head())

print("Renewals Preview:")
display(renewals.head())

print("Events Preview:")
display(events.head())

Members Preview:


,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023


Renewals Preview:


,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9


Events Preview:


,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13


## 4. Metadata Exploration & Ratio Checks
Inspect schema, check unique member distribution, calculate average events per member, and verify primary key uniqueness.

In [4]:
# Inspect data types, non-null counts, and memory usage for events
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 152057 entries, 0 to 152056
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   event_id     152057 non-null  str           
 1   member_id    152057 non-null  str           
 2   renewal_id   152057 non-null  str           
 3   event_time   152057 non-null  datetime64[us]
 4   event_type   152057 non-null  str           
 5   channel      152057 non-null  str           
 6   outcome      152057 non-null  str           
 7   detail_code  143033 non-null  str           
 8   amount       152057 non-null  int64         
 9   event_cost   152057 non-null  int64         
dtypes: datetime64[us](1), int64(2), str(7)
memory usage: 11.6 MB


In [5]:
# Inspect total rows and unique members in events
print("Total events count:", events.shape[0])
print("Unique member IDs in events:", events["member_id"].nunique())
display(events["member_id"].unique())

Total events count: 152057
Unique member IDs in events: 6380


<StringArray>
['MP000456', 'MP000612', 'MP001531', 'MP002497', 'MP001821', 'MP002061',
 'MP000447', 'MP001371', 'MP001411', 'MP002032',
 ...
 'MP006299', 'MP004647', 'MP005542', 'MP005923', 'MP005500', 'MP005829',
 'MP005848', 'MP005450', 'MP005307', 'MP005809']
Length: 6380, dtype: str

In [6]:
# Calculate the average number of events per unique member
avg_events_per_member = round((events.shape[0] / events["member_id"].nunique()), 2)
print("Average events per member:", avg_events_per_member)

Average events per member: 23.83


### Verify Primary Key Uniqueness (`event_id`)
Check whether `event_id` is globally unique across all event records (a ratio of 1.0 indicates a strictly unique primary key).

In [7]:
# Verify uniqueness of event_id
event_id_uniqueness_ratio = events["event_id"].nunique() / events.shape[0]
print("event_id uniqueness ratio (1.0 = strictly unique):", event_id_uniqueness_ratio)

event_id uniqueness ratio (1.0 = strictly unique): 1.0


## 5. Duplicate Detection & Data Cleaning
Identify duplicate events matching on `['member_id', 'renewal_id', 'event_time']` and remove all records associated with those affected members across all three tables.

In [59]:
# Define composite key columns to check for duplicate event occurrences
cols = ["member_id", "renewal_id", "event_time"]

# Calculate the total number of duplicate groupings
duplicate_groups_count = (events.groupby(cols).size() > 1).sum()
print("Total duplicate event groups found:", duplicate_groups_count)

Total duplicate event groups found: 19


In [60]:
# Group events by composite key and filter those appearing more than once
# Use .reset_index() to convert the Series into a DataFrame with accessible columns
group = events.groupby(cols).size()
group = group[group > 1].reset_index(name="event_count")

print("Duplicate event groupings:")
display(group)

Duplicate event groupings:


,member_id,renewal_id,event_time,event_count
0,MP000056,RN003199,2025-09-04 19:30:00,2
1,MP000152,RN004344,2025-08-01 13:59:00,2
2,MP000391,RN007248,2026-08-04 13:56:00,2
3,MP000408,RN003661,2025-09-03 11:58:00,2
4,MP000650,RN003470,2025-09-20 10:00:00,2
5,MP001695,RN007086,2026-09-07 10:00:00,2
6,MP002242,RN007127,2026-08-21 11:23:00,2
7,MP002629,RN009132,2026-08-09 20:20:00,2
8,MP002719,RN005913,2025-07-23 09:23:00,2
9,MP002900,RN005507,2025-08-09 20:07:00,2


In [61]:
# Extract the list of unique member IDs associated with duplicate events
duplicate_member_ids = group["member_id"].unique().tolist()
print(f"Total member IDs with duplicate events to remove: {len(duplicate_member_ids)}")

# Remove duplicate member IDs entirely across events, members, and renewals tables
events = events[~events["member_id"].isin(duplicate_member_ids)]
members = members[~members["member_id"].isin(duplicate_member_ids)]
renewals = renewals[~renewals["member_id"].isin(duplicate_member_ids)]

# Verify cleaned row counts and remaining unique members
print("Post-cleaning dataset summary:")
print(f"Events:   {events.shape[0]} rows (Unique members: {events['member_id'].nunique()})")
print(f"Members:  {members.shape[0]} rows (Unique members: {members['member_id'].nunique()})")
print(f"Renewals: {renewals.shape[0]} rows (Unique members: {renewals['member_id'].nunique()})")

Total member IDs with duplicate events to remove: 19
Post-cleaning dataset summary:
Events:   151472 rows (Unique members: 6361)
Members:  8561 rows (Unique members: 8561)
Renewals: 11542 rows (Unique members: 6361)


## 6. Primary & Candidate Key Validation
Define a reusable helper function `check_key` to confirm whether a given column or set of columns uniquely identifies rows in a DataFrame.

In [11]:
def check_key(df: pd.DataFrame, cols: list) -> bool:
    """
    Check whether specified column(s) uniquely identify each row in df.
    Returns True if no duplicates exist, False otherwise.
    """
    group_counts = df.groupby(cols).size()
    return (group_counts > 1).sum() == 0

# Validate primary key for members table
is_member_key_unique = check_key(members, ["member_id"])
print("Is ['member_id'] a unique primary key in members?:", is_member_key_unique)

# Validate composite key for renewals table
is_renewal_key_unique = check_key(renewals, ["member_id", "renewal_number"])
print("Is ['member_id', 'renewal_number'] a unique key in renewals?:", is_renewal_key_unique)

Is ['member_id'] a unique primary key in members?: True
Is ['member_id', 'renewal_number'] a unique key in renewals?: True


## 7. Member Event Timeline Inspection
Inspect the chronological event history for a specific member (`MP000456`) to observe sequential behavior over time.

In [12]:
# Filter events for a sample member and sort chronologically by event_time
sample_member_id = "MP000456"
member_timeline = events[events["member_id"] == sample_member_id].sort_values(by="event_time")

print(f"Event timeline for member: {sample_member_id} ({len(member_timeline)} total events)")
display(member_timeline.head(10))

Event timeline for member: MP000456 (51 total events)


,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10
199,EV0023867,MP000456,RN001802,2024-07-07 14:00:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,54
232,EV0023864,MP000456,RN001802,2024-07-07 20:24:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,27
1532,EV0023866,MP000456,RN001802,2024-07-16 18:12:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,40
2378,EV0023865,MP000456,RN001802,2024-07-20 10:03:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,20
2485,EV0023863,MP000456,RN001802,2024-07-20 14:53:00,BENEFIT_USED,WEB,SUCCESS,MEMBER_OFFER,0,11
2567,EV0023862,MP000456,RN001802,2024-07-20 18:16:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,65
2868,EV0023860,MP000456,RN001802,2024-07-21 17:20:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,54
3394,EV0023858,MP000456,RN001802,2024-07-23 15:24:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,26
5464,EV0023868,MP000456,RN001802,2024-07-29 08:10:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,19


## 8. Cohort Sampling (10% Random Sample)
Draw a 10% random sample of members, and subset `events` and `renewals` using the sampled member IDs to preserve relational consistency.

In [13]:
# Sample 10% of member IDs from cleaned members table (random_state ensures reproducibility)
sample_member_ids = members["member_id"].sample(frac=0.1, random_state=42).tolist()
print(f"Sampled {len(sample_member_ids)} member IDs out of {len(members)} total.")

# Filter all three tables using the sampled member IDs
sample_members = members[members["member_id"].isin(sample_member_ids)]
sample_renewals = renewals[renewals["member_id"].isin(sample_member_ids)]
sample_events = events[events["member_id"].isin(sample_member_ids)]

print(f"Sampled Members shape:  {sample_members.shape}")
print(f"Sampled Renewals shape: {sample_renewals.shape}")
print(f"Sampled Events shape:   {sample_events.shape}")

Sampled 856 member IDs out of 8561 total.
Sampled Members shape:  (856, 9)
Sampled Renewals shape: (1159, 9)
Sampled Events shape:   (15402, 10)


In [62]:
# Preview the sampled datasets
print("Sampled Members Preview:")
display(sample_members.head())

print("Sampled Renewals Preview:")
display(sample_renewals.head())

print("Sampled Events Preview:")
display(sample_events.head())

Sampled Members Preview:


,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
10,MP004559,2025-09-22,PLUS_STANDARD,1499,STORE,Mumbai,PAID,PAID,2025
21,MP005691,2025-09-26,PLUS_STANDARD,1499,WEB,Hyderabad,PAID,EXPIRED,2025
49,MP006358,2025-10-07,PLUS_PREMIUM,2499,APP,Delhi,PAID,PAID,2025
52,MP002832,2024-09-30,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2024
56,MP000946,2023-09-21,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023


Sampled Renewals Preview:


,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
18,RN003591,MP001718,2,2025-09-25,2025-10-02,2499,0,2025,9
23,RN000987,MP001579,1,2024-09-03,2024-09-10,2499,250,2024,9
61,RN002912,MP001870,2,2025-09-13,2025-09-20,1499,0,2025,9
84,RN008372,MP003440,2,2026-09-23,2026-09-30,1499,0,2026,9


Sampled Events Preview:


,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55
6,EV0000753,MP001531,RN000060,2024-07-03 19:36:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,37
41,EV0024430,MP000453,RN001847,2024-07-04 17:01:00,BENEFIT_USED,WEB,SUCCESS,MEMBER_OFFER,0,25
42,EV0000750,MP001531,RN000060,2024-07-04 17:53:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,60
44,EV0011511,MP002436,RN000868,2024-07-04 18:40:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,53


In [63]:
events["event_time"] = pd.to_datetime(events["event_time"])

In [64]:
events["event_year"] = events["event_time"].dt.year
events.groupby('event_year').size()

event_year
2024    32224
2025    51085
2026    68163
dtype: int64

In [65]:
# Find Renewals
cond = (events["event_type"]=="PAYMENT_ATTEMPT")&(events["outcome"]=="SUCCESS")
events[cond].groupby('event_year').agg(
    members_renewed = ("member_id", "nunique"),
    total_renewals = ("renewal_id", "nunique"),

)

,members_renewed,total_renewals
event_year,,
2024,1993,1993
2025,3188,3188
2026,3813,3813


In [66]:
events.groupby('event_year').agg(members_total = ("member_id", "nunique"),)

,members_total
event_year,
2024,2493
2025,3986
2026,5063


In [67]:
cond= (events['event_type']=='PAYMENT_ATTEMPT')&(events['outcome']=='SUCCESS')

events['is_renewed'] = np.where(cond,1,0)

In [68]:
events.groupby('event_year').agg(members_total = ("member_id", "nunique"),
                                 total_renewed = ("is_renewed", "sum"),)

,members_total,total_renewed
event_year,,
2024,2493,1993
2025,3986,3188
2026,5063,3813


*“More members are renewing, but I want to improve the chance that a member renews. I also don’t want us to achieve that simply by giving away membership fees.”*

1. Establish the metric “Our primary metric is the renewal rate.”
- Renewal rate = successful renewals ÷ eligible renewal opportunities

| Rule | Why we need it |
|---|---|
| Compare September expiries across years | Keep the reporting periods comparable |
| Include only completed renewal windows | Give every opportunity its full chance to renew |
| Count success within the valid payment window | Apply one consistent renewal definition |
| Count each opportunity once | Retries must not inflate renewal counts |

In [69]:
# deadline_not_completed = renewals["renewal_deadline"]>"2025-09-09"

renewals.info()


<class 'pandas.DataFrame'>
Index: 11542 entries, 0 to 11579
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   renewal_id        11542 non-null  str           
 1   member_id         11542 non-null  str           
 2   renewal_number    11542 non-null  int64         
 3   expiry_date       11542 non-null  datetime64[us]
 4   renewal_deadline  11542 non-null  datetime64[us]
 5   renewal_fee       11542 non-null  int64         
 6   offered_discount  11542 non-null  int64         
 7   expiry_year       11542 non-null  int32         
 8   expiry_month      11542 non-null  int32         
dtypes: datetime64[us](2), int32(2), int64(3), str(2)
memory usage: 811.5 KB


In [80]:
renewals["is_eligible"] = np.where(renewals["renewal_deadline"]< pd.to_datetime("2026-09-09"),1,0)

renewals

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month,is_eligible
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9,1
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9,0
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9,1
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9,0
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9,1
...,...,...,...,...,...,...,...,...,...,...
11575,RN003972,MP001840,2,2025-09-04,2025-09-11,2499,0,2025,9,1
11576,RN000572,MP002177,1,2024-09-03,2024-09-10,1499,0,2024,9,1
11577,RN009390,MP003830,2,2026-09-10,2026-09-17,1499,0,2026,9,0
11578,RN010994,MP005933,1,2026-10-07,2026-10-14,1499,0,2026,10,0


In [82]:
# events.merge(renewals[['renewal_id', 'is_eligible']], on = 'renewal_id', how = 'left')
events_renewal = events.merge(renewals[['renewal_id', 'is_eligible', 'renewal_fee', 'renewal_number', 'offered_discount']], on = 'renewal_id', how = 'left')
events_renewal.head()

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,event_year,is_renewed,is_eligible,renewal_fee,renewal_number,offered_discount
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,2024,0,1,1499,1,0
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,2024,0,1,2499,1,0
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55,2024,0,1,2499,1,0
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,2024,0,1,1499,1,0
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,2024,0,1,1499,1,0


In [83]:
cond = (events_renewal['event_type'] == 'PAYMENT_ATTEMPT') & (events_renewal['outcome'] == 'SUCCESS') & (events_renewal['is_eligible'] == 1)


In [85]:
group = events_renewal[cond].groupby('event_year').agg(
    members_total = ("member_id", "nunique"),
    total_renewed = ("is_renewed", "sum"),
    cash_collected = ("event_cost", "sum"),
    fee_collected = ('renewal_fee', 'sum'),
    total_discount = ('offered_discount', 'sum')
)

Cost collected every month

In [86]:
events_renewal['event_year_month'] = events_renewal['event_time'].dt.strftime('%Y-%m')

In [87]:
events_renewal[cond].groupby('event_year_month').agg(members_total = ('member_id', 'nunique'), total_renewed = ('is_renewed', 'sum'), cash_collected = ('amount', 'sum'), fee_collected = ('renewal_fee', 'sum'), total_discount = ('offered_discount', 'sum'))

,members_total,total_renewed,cash_collected,fee_collected,total_discount
event_year_month,,,,,
2024-08,33,33,55817,56467,650
2024-09,1787,1787,3194913,3232713,37800
2024-10,173,173,313827,317327,3500
2025-08,65,65,122185,123435,1250
2025-09,2844,2844,5049256,5112156,62900
2025-10,279,279,495471,501221,5750
2026-08,34,34,60666,60966,300
2026-09,96,96,170604,171904,1300
